# SBP data acquisition

Use the saved calibration to visit every sub-thread and record complex VNA traces and measured bead positions. The implementation lives in `src/acquisition.py`; the motor and calibration modules are shared with the calibration notebook.

The default simulated run opens no instruments and stores **zero traces**, exercising the same scan and output format. For a real measurement, finish the calibration notebook first and edit your local `config/measurement_config.json`.

In [ ]:
from pathlib import Path
from src.acquisition import (
    MeasurementConfig, create_demo_config, load_calibration,
    plan_scan, acquire, summarize_run,
)

SIMULATE = True
VNA_TESTMODE = SIMULATE
CONFIG_FILE = (create_demo_config("outputs/acquisition") if SIMULATE
               else Path("config/measurement_config.json"))
config = MeasurementConfig.from_file(CONFIG_FILE)

## 1. Review settings and calibration

Configuration holds the motor port/direction/profile and travel limits, the VNA address/channel/ports/band/averages, scan spacing, settling time, selected sub-threads, calibration file and output directory. Distances are in metres; frequency settings are in Hz.

A timestamped run directory is created unless `output.directory` is set. Existing scan files are refused by default.

In [ ]:
calibration = load_calibration(config)
print(calibration.summary())
print("Motor limits:", config.stepper.min_position_steps, config.stepper.max_position_steps)
print(f"VNA: S{config.vna.port_a}{config.vna.port_b}, channel {config.vna.channel}, "
      f"{config.vna.start_freq_Hz/1e9:g}–{config.vna.stop_freq_Hz/1e9:g} GHz, "
      f"{config.vna.points} points, {config.vna.averages} averages")
print("Simulated motor:", SIMULATE, "VNA zero-trace mode:", VNA_TESTMODE)

## 2. Preview the scan

This step connects to no hardware. Target positions are measured along each sub-thread from its calibrated start, with its end margins removed. Positions written during acquisition use the actual reached motor step count and interpolated XYZ coordinates.

In [ ]:
targets = plan_scan(config)
print("Planned points:", len(targets))
print("thread  point  position [m]  target [steps]")
for sub, index, _, position, steps in targets[:20]:
    print(f"{sub.index:6d} {index:6d} {position:13.6f} {steps:15d}")

## 3. Acquire

For a real run, set `SIMULATE=False` and `VNA_TESTMODE=False` in the first cell and supply your instrument settings. The real VNA sweep is read back before scanning so the saved frequency axis matches the instrument.

Kernel **Interrupt** aborts acquisition; the motor and VNA are released by cleanup. The scan manifest records completion, interruption or failure and the number of finished measurements. After a motor emergency stop, its position must be re-established before another move.

In [ ]:
run_directory = acquire(
    config, simulate=SIMULATE, vna_testmode=VNA_TESTMODE,
    config_path=CONFIG_FILE,
)
print("Saved run:", run_directory)

## 4. Inspect the output

- `s11_traces.h5`: `frequencies_Hz` and complex `s_parameter`, with axes `(trace, frequency)`.
- `scan_log.csv` and `scan_log.jsonl`: positions, timestamps and trace indices.
- `scan_manifest.json`: configuration, calibration, scan status and counts.
- `vna_log.txt`: instrument communication log when using a real VNA.

The historical HDF5 filename is retained. The actual S-parameter is identified by its attributes (by default **S22**, ports 2/2), not by the filename.

In [ ]:
summarize_run(run_directory)